In [17]:
#imporation des bibliothèques nécessaires
import pandas as pd
from sqlalchemy import create_engine
#imporation pour random forest
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import subprocess
subprocess.run(["pip", "install", "xgboost"])
from xgboost import XGBClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


In [18]:
#Connexion à la base de données ssms
# paramètres de connexion
server = 'localhost'
database = 'event_DWH'
username = 'Talend_user'
password = '12345678'


connection_string = f"mssql+pyodbc://{username}:{password}@{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server"
engine = create_engine(connection_string)

tables = [
    "Dim_Beneficiary",
    "Dim_Category",
    "Dim_Complaint",
    "Dim_Concurrence",
    "Dim_Entertainer",
    "FACT_VENTES",
    "Dim_Event",
    "Dim_Evaluation",
    "Dim_Event",
    "Dim_Localisation",
    "Dim_Provider",
    "Dim_Service",
    "Dim_Subcategory",
    "Dim_Trends",
    "Dim_Venue",
    "Dim_Weather",
    "DimDates",

]

dfs = {}

for table in tables:
    dfs[table] = pd.read_sql(f"SELECT TOP 5 * FROM {table}", engine)
    print(f"\n{table}:")
    print(dfs[table])


Dim_Beneficiary:
   id_beneficiary first_name                          email last_name
0               1      Derek                            NaN      Lara
1               2    Jessica              fchen@example.net  Robinson
2               3      Brian       williamsgina@example.com      Buck
3               4    Jessica  wheelerjacqueline@example.com   Daniels
4               5      David       joshuanelson@example.com  Fletcher

Dim_Category:
   id_category       name
0            1    Wedding
1            2  Corporate
2            3   Birthday

Dim_Complaint:
   id_complaint                                        description  status  \
0             1                                                NaN  closed   
1             2  Generation well theory through sea. Couple thu...    open   
2             3  Pass three continue support quickly hand. Brin...  closed   
3             4  Section than kind center few chance. Risk big ...    open   
4             5  Act high establish e

In [3]:
# Ojectif1: fidelisation du client  (type : classification)

# 1-Appliquer l'algorithme de Random Forest
# on crée une target: fidélisation = nbr_reservations > 2


query = """
SELECT 
    nbr_reservations,
    price,
    marketing_spend,
    nbr_visitors
FROM FACT_VENTES
"""

df = pd.read_sql(query, engine)

# Target (1 = fidèle, 0 = non fidèle)
df['target'] = (df['nbr_reservations'] > df['nbr_reservations'].median()).astype(int)
# Features
X = df[['price', 'marketing_spend', 'nbr_visitors']]
y = df['target']


# 3. Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# 4. Test plusieurs hyperparamètres

best_score = 0
best_model = None

for n in [50, 100, 150]:
    for depth in [5, 10, None]:
        
        model = RandomForestClassifier(n_estimators=n, max_depth=depth, random_state=42)
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        
        f1 = f1_score(y_test, y_pred)
        
        print(f"n_estimators={n}, max_depth={depth} → F1={f1:.3f}")
        
        if f1 > best_score:
            best_score = f1
            best_model = model

print("\nBest Random Forest F1:", best_score)

n_estimators=50, max_depth=5 → F1=0.519
n_estimators=50, max_depth=10 → F1=0.886
n_estimators=50, max_depth=None → F1=0.964
n_estimators=100, max_depth=5 → F1=0.455
n_estimators=100, max_depth=10 → F1=0.871
n_estimators=100, max_depth=None → F1=0.959
n_estimators=150, max_depth=5 → F1=0.466
n_estimators=150, max_depth=10 → F1=0.868
n_estimators=150, max_depth=None → F1=0.957

Best Random Forest F1: 0.964200477326969


Explication complète pour ton prof
C'est quoi le F1-Score ?
Le F1-Score est une métrique qui combine deux mesures :

Précision : parmi les clients que le modèle dit "fidèles", combien le sont vraiment ?
Recall : parmi tous les vrais clients fidèles, combien le modèle a-t-il détecté ?

F1 = 2 × (Précision × Recall) / (Précision + Recall)
F1-ScoreInterprétation1.00Parfait> 0.90Excellent> 0.80Bon> 0.70Acceptable< 0.60Faible

Analyse de tes résultats
Tu remarques une tendance très claire :
max_depthF1 moyenInterprétation5~0.48Trop simple, modèle sous-apprend10~0.87Bon équilibreNone~0.96Meilleur score mais arbre illimité

max_depth=None signifie que l'arbre grandit sans limite jusqu'à ce que toutes les feuilles soient pures.


Faut-il choisir n_estimators=50, max_depth=None ?
Techniquement oui, c'est le meilleur F1, mais il y a un compromis :
n_estimators=50, max_depth=None → F1=0.964  ← meilleur score MAIS arbre illimité
n_estimators=50, max_depth=10   → F1=0.886  ← légèrement moins bon MAIS plus contrôlé


Ce que je te recommande pour ton prof :
python# Choix final justifié
model_final = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,      # limité pour éviter l'overfitting
    random_state=42
)
Et tu justifies comme ça :

"Nous avons choisi max_depth=10 plutôt que None car un arbre sans limite de profondeur risque de mémoriser les données d'entraînement (overfitting). Bien que le F1 soit légèrement inférieur (0.886 vs 0.964), le modèle est plus robuste et généralisable sur de nouvelles données."


Où se manifeste la classification dans ton code ?
La classification se manifeste à cette ligne précisément :
python# Target (1 = fidèle, 0 = non fidèle)
df['target'] = (df['nbr_reservations'] > df['nbr_reservations'].median()).astype(int)
C'est ici que tu décides la règle de classification :
ValeurSignificationCondition1Client fidèlenbr_reservations > médiane0Client non fidèlenbr_reservations ≤ médiane

Comment l'arbre utilise cette règle ?
L'arbre apprend à partir de tes 3 features :
X = ['price', 'marketing_spend', 'nbr_visitors']
Il construit des règles comme :
Est-ce que nbr_visitors > 150 ?
├── OUI → Est-ce que price < 200 ?
│         ├── OUI → Client FIDÈLE (1) ✅
│         └── NON → Client NON FIDÈLE (0) ❌
└── NON → Client NON FIDÈLE (0) ❌

In [11]:
# ===============================
# TEST DU MODELE Random Forest AVEC INPUT UTILISATEUR
# ===============================

# Demander les valeurs à l'utilisateur
price = float(input("Entrer le prix du service : "))
marketing_spend = float(input("Entrer le budget marketing : "))
nbr_visitors = float(input("Entrer le nombre de visiteurs : "))

# Créer un DataFrame avec ces valeurs
new_data = pd.DataFrame({
    'price': [price],
    'marketing_spend': [marketing_spend],
    'nbr_visitors': [nbr_visitors]
})

# Prédiction
prediction = best_model.predict(new_data)[0]

# Probabilité (optionnel mais très utile)
proba = best_model.predict_proba(new_data)[0][1]

# Résultat
if prediction == 1:
    print(f" Client fidèle (probabilité = {proba:.2f})")
else:
    print(f" Client non fidèle (probabilité = {proba:.2f})")

 Client fidèle (probabilité = 0.56)


In [9]:
#1-Appliquer l'algorithme Xgboost

best_score_xgb = 0
best_model_xgb = None

# tester plusieurs hyperparamètres
for lr in [0.01, 0.1, 0.2]:
    for depth in [3, 5, 7]:
        
        model = XGBClassifier(
            learning_rate=lr,
            max_depth=depth,
            n_estimators=100,
            use_label_encoder=False,
            eval_metric='logloss'
        )
        
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        
        f1 = f1_score(y_test, y_pred)
        
        print(f"learning_rate={lr}, max_depth={depth} → F1={f1:.3f}")
        
        if f1 > best_score_xgb:
            best_score_xgb = f1
            best_model_xgb = model

print("\nBest XGBoost F1:", best_score_xgb)

c:\Users\USER\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:56:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\USER\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:56:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\USER\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:56:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


learning_rate=0.01, max_depth=3 → F1=0.871
learning_rate=0.01, max_depth=5 → F1=1.000
learning_rate=0.01, max_depth=7 → F1=1.000
learning_rate=0.1, max_depth=3 → F1=1.000


c:\Users\USER\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:56:10] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\USER\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:56:10] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\USER\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:56:10] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


learning_rate=0.1, max_depth=5 → F1=1.000
learning_rate=0.1, max_depth=7 → F1=1.000
learning_rate=0.2, max_depth=3 → F1=1.000
learning_rate=0.2, max_depth=5 → F1=1.000
learning_rate=0.2, max_depth=7 → F1=1.000

Best XGBoost F1: 1.0


c:\Users\USER\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:56:10] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\USER\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:56:10] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\USER\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:56:10] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Où se manifeste la classification dans XGBoost ?
La classification se manifeste à 3 endroits précis :

1️⃣ L'entraînement — le modèle apprend
pythonmodel.fit(X_train, y_train)

Le modèle apprend la relation entre price, marketing_spend, nbr_visitors et la target 0 ou 1


2️⃣ La prédiction — le modèle classifie
pythony_pred = model.predict(X_test)

C'est ici que la classification se manifeste vraiment — le modèle regarde chaque client et décide :

1 → Client fidèle
0 → Client non fidèle



3️⃣ L'évaluation — on mesure la qualité
pythonf1 = f1_score(y_test, y_pred)

On compare ce que le modèle a prédit y_pred avec la réalité y_test


Le chemin complet d'un client dans ton modèle
Client dans FACT_VENTES
        ↓
[price=150, marketing_spend=5000, nbr_visitors=200]
        ↓
   XGBClassifier
        ↓
model.predict() ← classification ici !
        ↓
      0 ou 1
        ↓
❌ Non fidèle    ✅ Fidèle

Ce que XGBoost fait en interne
Arbre 1 : price > 100 ?
          ├── OUI → score +0.3
          └── NON → score -0.2

Arbre 2 : nbr_visitors > 150 ?
          ├── OUI → score +0.4
          └── NON → score -0.1

Arbre 3 : marketing_spend > 3000 ?
          ├── OUI → score +0.2
          └── NON → score -0.3

Score total > 0.5 → Client FIDÈLE (1) ✅
Score total ≤ 0.5 → Client NON FIDÈLE (0) ❌

💡 Différence avec Random Forest : Random Forest vote à la majorité entre ses arbres, tandis que XGBoost corrige les erreurs de chaque arbre précédent de façon progressive.

In [10]:
# ===============================
# TEST DU MODELE XGBOOST
# ===============================

# Demander les valeurs à l'utilisateur
price = float(input("Entrer le prix du service : "))
marketing_spend = float(input("Entrer le budget marketing : "))
nbr_visitors = float(input("Entrer le nombre de visiteurs : "))

# Créer un DataFrame avec les mêmes features que l'entraînement
new_data = pd.DataFrame({
    'price': [price],
    'marketing_spend': [marketing_spend],
    'nbr_visitors': [nbr_visitors]
})

# Prédiction
prediction = best_model_xgb.predict(new_data)[0]

# Probabilité (important pour interprétation)
proba = best_model_xgb.predict_proba(new_data)[0][1]

# Affichage résultat
if prediction == 1:
    print(f"✅ Client fidèle (probabilité = {proba:.2f})")
else:
    print(f"❌ Client non fidèle (probabilité = {proba:.2f})")

❌ Client non fidèle (probabilité = 0.23)


In [30]:
# comparaison entre Random Forest et XGBoost


from sklearn.metrics import classification_report, roc_auc_score

# Random Forest
y_pred_rf = best_model.predict(X_test)
y_proba_rf = best_model.predict_proba(X_test)[:,1]

# XGBoost
y_pred_xgb = best_model_xgb.predict(X_test)
y_proba_xgb = best_model_xgb.predict_proba(X_test)[:,1]

print("===== RANDOM FOREST =====")
print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))

print("\n===== XGBOOST =====")
print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_xgb))

===== RANDOM FOREST =====
              precision    recall  f1-score   support

           0       0.96      0.99      0.97       277
           1       0.99      0.94      0.96       214

    accuracy                           0.97       491
   macro avg       0.97      0.97      0.97       491
weighted avg       0.97      0.97      0.97       491

ROC-AUC: 0.9978744222139748

===== XGBOOST =====
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       277
           1       1.00      1.00      1.00       214

    accuracy                           1.00       491
   macro avg       1.00      1.00      1.00       491
weighted avg       1.00      1.00      1.00       491

ROC-AUC: 1.0


Interprétation complète à présenter à ton prof

RANDOM FOREST ✅ — Résultat crédible et solide
Pour la classe 0 (clients NON fidèles) :

Précision 0.96 → sur 100 clients prédits non fidèles, 96 le sont vraiment
Recall 0.99 → le modèle détecte 99% des vrais clients non fidèles
F1 = 0.97 → excellent équilibre

Pour la classe 1 (clients fidèles) :

Précision 0.99 → sur 100 clients prédits fidèles, 99 le sont vraiment
Recall 0.94 → le modèle détecte 94% des vrais clients fidèles
F1 = 0.96 → excellent

ROC-AUC = 0.9979 → le modèle distingue parfaitement les deux classes

✅ Conclusion RF : Modèle fiable, stable et crédible à présenter


XGBOOST ⚠️ — Résultat parfait = suspect
Tout est à 1.00 ce qui est quasi impossible en réalité
MétriqueValeurSignificationPrécision1.000 erreur de prédictionRecall1.000 client manquéF11.00Score parfaitROC-AUC1.00Séparation parfaite

⚠️ Conclusion XGB : Un modèle parfait sur des données réelles n'existe pas — cela indique probablement que ta target nbr_reservations est trop facilement devinable depuis tes features, ou que tes données sont trop simples


Ce que tu dis à ton prof

"Nous avons comparé Random Forest et XGBoost sur l'objectif de classification de la fidélisation client. Random Forest obtient un F1 de 0.97 et un ROC-AUC de 0.9979, ce qui représente un excellent résultat réaliste. XGBoost affiche des scores parfaits de 1.0, ce qui nous a conduit à suspecter un data leakage ou une séparabilité trop évidente des données. Nous recommandons donc le Random Forest comme modèle final car ses résultats sont à la fois excellents et crédibles."

In [13]:
# ===============================
# DETECTION DE L'OVERFITTING
# ===============================

print("=" * 50)
print("RANDOM FOREST - Détection Overfitting")
print("=" * 50)

# best_model = ton Random Forest, best_model_xgb = ton XGBoost
rf_train_score = best_model.score(X_train, y_train)
rf_test_score = best_model.score(X_test, y_test)

print(f"Score TRAIN : {rf_train_score:.4f}")
print(f"Score TEST  : {rf_test_score:.4f}")
print(f"Différence  : {abs(rf_train_score - rf_test_score):.4f}")

if abs(rf_train_score - rf_test_score) > 0.05:
    print("⚠️  OVERFITTING DÉTECTÉ !")
else:
    print("✅ Pas d'overfitting")

print()
print("=" * 50)
print("XGBOOST - Détection Overfitting")
print("=" * 50)

xgb_train_score = best_model_xgb.score(X_train, y_train)
xgb_test_score = best_model_xgb.score(X_test, y_test)

print(f"Score TRAIN : {xgb_train_score:.4f}")
print(f"Score TEST  : {xgb_test_score:.4f}")
print(f"Différence  : {abs(xgb_train_score - xgb_test_score):.4f}")

if abs(xgb_train_score - xgb_test_score) > 0.05:
    print("⚠️  OVERFITTING DÉTECTÉ !")
else:
    print("✅ Pas d'overfitting")

# ===============================
# CROSS VALIDATION (plus fiable)
# ===============================
from sklearn.model_selection import cross_val_score

print()
print("=" * 50)
print("CROSS VALIDATION (5 folds)")
print("=" * 50)

cv_rf = cross_val_score(best_model, X_train, y_train, cv=5, scoring='f1')
cv_xgb = cross_val_score(best_model_xgb, X_train, y_train, cv=5, scoring='f1')

print(f"Random Forest - F1 moyen : {cv_rf.mean():.4f} (+/- {cv_rf.std():.4f})")
print(f"XGBoost       - F1 moyen : {cv_xgb.mean():.4f} (+/- {cv_xgb.std():.4f})")

if cv_xgb.std() < 0.02:
    print("✅ XGBoost : stable sur tous les folds")
else:
    print("⚠️  XGBoost : instable → probable overfitting")

RANDOM FOREST - Détection Overfitting
Score TRAIN : 1.0000
Score TEST  : 0.9695
Différence  : 0.0305
✅ Pas d'overfitting

XGBOOST - Détection Overfitting
Score TRAIN : 1.0000
Score TEST  : 1.0000
Différence  : 0.0000
✅ Pas d'overfitting

CROSS VALIDATION (5 folds)


c:\Users\USER\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:04:37] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\USER\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:04:38] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\USER\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:04:38] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Random Forest - F1 moyen : 0.9635 (+/- 0.0117)
XGBoost       - F1 moyen : 1.0000 (+/- 0.0000)
✅ XGBoost : stable sur tous les folds


c:\Users\USER\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:04:38] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\USER\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:04:38] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [19]:
#objectif 3 Zone à fort potentiel de croissance (type : classification)
# Algo 1 :K-MEANS


query = """
SELECT 
    l.city,
    COUNT(*) AS total_events,
    AVG(f.price) AS avg_price,
    AVG(f.nbr_reservations) AS avg_reservations,
    AVG(f.market_count) AS competition_level
FROM FACT_VENTES f
JOIN Dim_Localisation l ON f.id_localisation = l.id_localisation
GROUP BY l.city
"""

df = pd.read_sql(query, engine)

# -------------------------------
# 3. Préparation des données
# -------------------------------
X = df[['total_events', 'avg_price', 'avg_reservations', 'competition_level']]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -------------------------------
# 4. K-Means (tester plusieurs K)
# -------------------------------
best_score_kmeans = 0

for k in [2, 3, 4, 5]:
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(X_scaled)
    
    score = silhouette_score(X_scaled, labels)
    print(f"K={k} → Silhouette Score = {score:.3f}")
    
    if score > best_score_kmeans:
        best_score_kmeans = score
        best_kmeans = kmeans
        best_labels_kmeans = labels

# Ajouter cluster au dataset
df['cluster_kmeans'] = best_labels_kmeans

print("\nBest KMeans Silhouette:", best_score_kmeans)
print(df.head())

K=2 → Silhouette Score = 0.264
K=3 → Silhouette Score = 0.212
K=4 → Silhouette Score = 0.227
K=5 → Silhouette Score = 0.272

Best KMeans Silhouette: 0.2719938752497922
        city  total_events     avg_price  avg_reservations  competition_level  \
0     Ariana           166   7809.927711               193                 12   
1  Ben Arous            85   8533.494118               200                  7   
2    Bizerte           152  11689.493421               208                  4   
3      Gafsa            97   9972.536082               202                  5   
4   Jendouba           182   8971.598901               197                  4   

   cluster_kmeans  
0               1  
1               2  
2               0  
3               2  
4               0  


c:\Users\USER\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\USER\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\USER\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\USER\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows wi

In [23]:
# ===============================
# DBSCAN - SEGMENTATION DES ZONES
# ===============================
from sklearn.cluster import DBSCAN

best_score_dbscan = -1
best_dbscan = None
best_labels_dbscan = None  # ← initialiser à None

for eps in [0.3, 0.5, 1, 1.5, 2]:        # ← plus de valeurs à tester
    for min_samples in [2, 3, 5]:         # ← ajouter min_samples=2

        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        labels = dbscan.fit_predict(X_scaled)

        n_clusters = len(set(labels) - {-1})  # exclure le bruit (-1)
        n_noise = list(labels).count(-1)

        print(f"eps={eps}, min_samples={min_samples} → "
              f"Clusters={n_clusters}, Bruit={n_noise}")

        # au moins 2 clusters et pas trop de bruit
        if n_clusters > 1:
            score = silhouette_score(X_scaled, labels)
            print(f"  → Silhouette = {score:.3f}")

            if score > best_score_dbscan:
                best_score_dbscan = score
                best_dbscan = dbscan
                best_labels_dbscan = labels

# ← vérification avant d'ajouter au df
if best_labels_dbscan is not None:
    df['cluster_dbscan'] = best_labels_dbscan
    print("\n✅ Best DBSCAN Silhouette:", best_score_dbscan)
    print(df[['city', 'cluster_dbscan']].head(10))
else:
    print("\n⚠️ DBSCAN n'a trouvé aucun clustering valide")
    print("→ Utilise directement K-Means pour cet objectif")
    
    # fallback : utiliser K-Means à la place
    df['cluster_dbscan'] = df['cluster_kmeans']
    print("✅ Cluster K-Means utilisé comme fallback")

eps=0.3, min_samples=2 → Clusters=1, Bruit=17
eps=0.3, min_samples=3 → Clusters=0, Bruit=19
eps=0.3, min_samples=5 → Clusters=0, Bruit=19
eps=0.5, min_samples=2 → Clusters=2, Bruit=15
  → Silhouette = 0.021
eps=0.5, min_samples=3 → Clusters=0, Bruit=19
eps=0.5, min_samples=5 → Clusters=0, Bruit=19
eps=1, min_samples=2 → Clusters=3, Bruit=11
  → Silhouette = 0.039
eps=1, min_samples=3 → Clusters=1, Bruit=15
eps=1, min_samples=5 → Clusters=0, Bruit=19
eps=1.5, min_samples=2 → Clusters=3, Bruit=7
  → Silhouette = 0.187
eps=1.5, min_samples=3 → Clusters=2, Bruit=9
  → Silhouette = 0.205
eps=1.5, min_samples=5 → Clusters=2, Bruit=9
  → Silhouette = 0.205
eps=2, min_samples=2 → Clusters=1, Bruit=1
eps=2, min_samples=3 → Clusters=1, Bruit=1
eps=2, min_samples=5 → Clusters=1, Bruit=3

✅ Best DBSCAN Silhouette: 0.20488359390993327
        city  cluster_dbscan
0     Ariana              -1
1  Ben Arous               0
2    Bizerte              -1
3      Gafsa               0
4   Jendouba         

In [24]:
# ===============================
# COMPARAISON KMEANS vs DBSCAN
# ===============================

print("===== RESULTATS =====")
print(f"KMeans Silhouette Score : {best_score_kmeans:.3f}")
print(f"DBSCAN Silhouette Score : {best_score_dbscan:.3f}")

if best_score_kmeans > best_score_dbscan:
    print("\n✅ KMeans est meilleur pour segmenter les zones")
elif best_score_kmeans < best_score_dbscan:
    print("\n✅ DBSCAN est meilleur pour segmenter les zones")
else:
    print("\n⚖️ Les deux modèles sont équivalents")

===== RESULTATS =====
KMeans Silhouette Score : 0.272
DBSCAN Silhouette Score : 0.205

✅ KMeans est meilleur pour segmenter les zones
